# Phase 3 — PEFT Fine-Tuning

In this notebook, we fine-tune **Voxtral-mini-3B** using **Parameter-Efficient Fine-Tuning (PEFT)** with **LoRA adapters**.

## Objective
- Improve emotion recognition performance compared to **Phase 2 (zero-shot inference)**
- Keep the training setup lightweight:
  - Base model loaded in 8-bit
  - Only LoRA adapters are trainable

## Task Definition
- **Input**: Speech audio + natural language instruction
- **Output**: Exactly one emotion label

This phase frames emotion recognition as an **instruction-conditioned generation task**, consistent with modern multimodal large language models.

## Dataset
- Emotion Speech Dataset (ESD)
- English speakers only (IDs 0011–0020)
- Emotion set:
  - Angry
  - Happy
  - Sad
  - Neutral  
- The emotion *Surprise* is excluded to keep label consistency across datasets.

## Important Design Choice
We intentionally **reuse the same chat prompt format from Phase 2** to ensure that fine-tuning and inference share the same input distribution.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Setup and Reproducibility

This section:
- Imports all core libraries used throughout the notebook
- Sets random seeds to ensure reproducibility
- Defines key filesystem paths used for:
  - EmoBox metadata
  - ESD audio files
  - local model checkpoints

We also detect whether a GPU is available and store the result in `device`.

In [2]:
import os
import sys
import json
import random
import numpy as np
import torch
import soundfile as sf
from datasets import Dataset
from transformers import (
    AutoProcessor,
    VoxtralForConditionalGeneration,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Paths
ADSP_ROOT = "/content/drive/MyDrive/adsp"
EMOBOX_ROOT = os.path.join(ADSP_ROOT, "EmoBox")

DATA_DIR = ADSP_ROOT                        # where "downloads/..." lives
META_DIR = os.path.join(EMOBOX_ROOT, "data")# parent dir containing esd/

print("ADSP_ROOT :", ADSP_ROOT)
print("EMOBOX_ROOT:", EMOBOX_ROOT)
print("DATA_DIR  :", DATA_DIR)
print("META_DIR  :", META_DIR)

assert os.path.exists(ADSP_ROOT), "ADSP_ROOT not found"
assert os.path.exists(EMOBOX_ROOT), "EMOBOX_ROOT not found"
assert os.path.exists(os.path.join(META_DIR, "esd")), "ESD metadata folder not found"

Device: cpu
ADSP_ROOT : /content/drive/MyDrive/adsp
EMOBOX_ROOT: /content/drive/MyDrive/adsp/EmoBox
DATA_DIR  : /content/drive/MyDrive/adsp
META_DIR  : /content/drive/MyDrive/adsp/EmoBox/data


## Loading the Emotion Speech Dataset (ESD)

We use **EmoBox**, a unified dataset loader, to load the ESD dataset.

Key points:
- We rely on the official train/test splits provided by EmoBox
- Audio paths are resolved relative to the `downloads/` directory
- Metadata is read from EmoBox JSON files

At this stage, no filtering is applied yet.

In [3]:
# ============================================================
# Load ESD dataset correctly with EmoBox
# ============================================================

import os
import sys

# ------------------------------------------------------------
# Set working directory
# ------------------------------------------------------------
os.chdir("/content/drive/MyDrive/adsp")
print("Current working directory:", os.getcwd())

# ------------------------------------------------------------
# Add EmoBox to Python path
# ------------------------------------------------------------
EMOBOX_ROOT = "/content/drive/MyDrive/adsp/EmoBox"
sys.path.append(EMOBOX_ROOT)

# ------------------------------------------------------------
# Define dataset paths
# ------------------------------------------------------------
DATA_DIR = "/content/drive/MyDrive/adsp"              # contains downloads/
META_DIR = "/content/drive/MyDrive/adsp/EmoBox/data"  # contains esd/, iemocap/

# Sanity checks
assert os.path.exists(DATA_DIR), "DATA_DIR does not exist"
assert os.path.exists(META_DIR), "META_DIR does not exist"
assert os.path.exists(os.path.join(DATA_DIR, "downloads", "esd")), "ESD audio not found"
assert os.path.exists(os.path.join(META_DIR, "esd")), "ESD metadata not found"

print("Paths verified.")

# ------------------------------------------------------------
# Load ESD using EmoBox
# ------------------------------------------------------------
from EmoBox.EmoDataset import EmoDataset

esd_train = EmoDataset(
    dataset="esd",
    data_dir=DATA_DIR,
    meta_data_dir=META_DIR,
    split="train"
)

esd_test = EmoDataset(
    dataset="esd",
    data_dir=DATA_DIR,
    meta_data_dir=META_DIR,
    split="test"
)

# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------
print("ESD train samples:", len(esd_train))
print("ESD test samples :", len(esd_test))

Current working directory: /content/drive/.shortcut-targets-by-id/1gH4RO3pZc-gbQXU4xk8xp-4rwW5XozbA/adsp
Paths verified.
since there is no official valid data, use random split for train valid split, with a ratio of [80, 20]
load in 28000 samples, only 28000 exists in data dir /content/drive/MyDrive/adsp/EmoBox/data
load in 7000 samples, only 7000 exists in data dir /content/drive/MyDrive/adsp/EmoBox/data
Num. training samples 28000
Num. valid samples 0
Num. test samples 7000
Using label_map {'Neutral': 'Neutral', 'Angry': 'Angry', 'Happy': 'Happy', 'Sad': 'Sad', 'Surprise': 'Surprise'}
since there is no official valid data, use random split for train valid split, with a ratio of [80, 20]
load in 28000 samples, only 28000 exists in data dir /content/drive/MyDrive/adsp/EmoBox/data
load in 7000 samples, only 7000 exists in data dir /content/drive/MyDrive/adsp/EmoBox/data
Num. training samples 28000
Num. valid samples 0
Num. test samples 7000
Using label_map {'Neutral': 'Neutral', 'Angry'

## Restricting the Dataset to English Speakers

ESD contains recordings in multiple languages.
To avoid introducing linguistic confounds, we restrict the dataset to **English speakers only**.

Selected speaker IDs:
- 0011 to 0020 (inclusive)

This ensures that:
- The spoken language matches the instruction language
- The model does not learn spurious language-emotion correlations

In [4]:
EN_SPEAKERS = {f"{i:04d}" for i in range(11, 21)}  # 0011..0020

def filter_esd_english(dataset):
    filtered = []
    for item in dataset.data_list:
        # wav looks like: downloads/esd/0001/Neutral/0001_000001.wav
        speaker = item["wav"].split("/")[2]
        if speaker in EN_SPEAKERS:
            filtered.append(item)
    dataset.data_list = filtered
    return dataset

esd_train = filter_esd_english(esd_train)
esd_test  = filter_esd_english(esd_test)

print("ESD train (English-only):", len(esd_train))
print("ESD test  (English-only):", len(esd_test))

ESD train (English-only): 12250
ESD test  (English-only): 5250


## Defining the Final Emotion Label Space

For this project, we focus on a **4-class emotion classification task**:

- Angry
- Happy
- Sad
- Neutral

The emotion *Surprise* is excluded to:
- Maintain consistency with IEMOCAP
- Simplify evaluation and comparison across phases

Emotion labels are normalized to have consistent capitalization.

In [5]:
EMOTIONS = ["Angry", "Happy", "Sad", "Neutral"]

def keep_4class(dataset):
    dataset.data_list = [
        {**x, "emo": x["emo"].capitalize()}
        for x in dataset.data_list
        if x["emo"].capitalize() in EMOTIONS
    ]
    return dataset

esd_train = keep_4class(esd_train)
esd_test  = keep_4class(esd_test)

print("ESD train (4-class):", len(esd_train))
print("ESD test  (4-class):", len(esd_test))

ESD train (4-class): 9800
ESD test  (4-class): 4200


## Building Instruction-Based Training Records

Instead of treating emotion recognition as a traditional classification task,
we frame it as **instruction-conditioned generation**.

Each training example consists of:
1. A **system message** defining the assistant’s role
2. A **user message** containing:
   - an audio reference (`audio_url`)
   - a textual instruction
3. A **target output**: the correct emotion label

### Why this design?
- Voxtral is a multimodal generative model
- This setup mirrors how the model is used at inference time
- It avoids mismatches between training and evaluation

In [6]:
# ============================================================
# Build Conversation-Based Training Records (Arrow-safe)
# ============================================================

import json

SYSTEM_PROMPT = "You are a careful and concise emotion classification assistant."

INSTRUCTION_PROMPT = (
    "You are an expert at recognizing emotions from speech. "
    "Listen to the audio and output only one label: "
    "Angry, Happy, Sad, or Neutral."
)

def build_conversations(dataset):
    records = []
    for item in dataset.data_list:
        conversation = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "audio_url",
                        "audio_url": "file://" + os.path.join(DATA_DIR, item["wav"]),
                    },
                    {
                        "type": "text",
                        "text": INSTRUCTION_PROMPT,
                    },
                ],
            },
        ]

        records.append({
            # 🔑 store conversation as JSON STRING
            "conversation": json.dumps(conversation),
            "label": item["emo"],
        })

    return records


train_records = build_conversations(esd_train)
test_records  = build_conversations(esd_test)

train_records[0]

{'conversation': '[{"role": "system", "content": "You are a careful and concise emotion classification assistant."}, {"role": "user", "content": [{"type": "audio_url", "audio_url": "file:///content/drive/MyDrive/adsp/downloads/esd/0012/Neutral/0012_000001.wav"}, {"type": "text", "text": "You are an expert at recognizing emotions from speech. Listen to the audio and output only one label: Angry, Happy, Sad, or Neutral."}]}]',
 'label': 'Neutral'}

## Creating Hugging Face Datasets

We convert the prepared training records into Hugging Face `Dataset` objects.

Important note:
- Conversations are stored as JSON strings
- They are decoded back into structured form inside the data collator

This design avoids limitations of Apache Arrow with nested Python objects.

In [7]:
from datasets import Dataset

train_ds = Dataset.from_list(train_records)
test_ds  = Dataset.from_list(test_records)

print(train_ds)
print(train_ds[0])

Dataset({
    features: ['conversation', 'label'],
    num_rows: 9800
})
{'conversation': '[{"role": "system", "content": "You are a careful and concise emotion classification assistant."}, {"role": "user", "content": [{"type": "audio_url", "audio_url": "file:///content/drive/MyDrive/adsp/downloads/esd/0012/Neutral/0012_000001.wav"}, {"type": "text", "text": "You are an expert at recognizing emotions from speech. Listen to the audio and output only one label: Angry, Happy, Sad, or Neutral."}]}]', 'label': 'Neutral'}


In [8]:
# ============================================================
# Dummy Test 1: Dataset sanity check
# ============================================================

ex = train_ds[0]
print("Keys:", ex.keys())
print("Label:", ex["label"])
print("Conversation preview:", ex["conversation"][:200], "...")

Keys: dict_keys(['conversation', 'label'])
Label: Neutral
Conversation preview: [{"role": "system", "content": "You are a careful and concise emotion classification assistant."}, {"role": "user", "content": [{"type": "audio_url", "audio_url": "file:///content/drive/MyDrive/adsp/d ...


In [9]:
from collections import Counter

# if using HF datasets with "label"
print("Train:", Counter(train_ds["label"]))
print("Test :", Counter(test_ds["label"]))

Train: Counter({'Neutral': 2450, 'Angry': 2450, 'Happy': 2450, 'Sad': 2450})
Test : Counter({'Neutral': 1050, 'Angry': 1050, 'Happy': 1050, 'Sad': 1050})


## Loading Voxtral-mini-3B

We load:
- The **Voxtral-mini-3B** model
- The corresponding processor

Key configuration choices:
- The base model is loaded in **8-bit precision**
- The model is prepared for fine-tuning
- Device placement is handled automatically

At this stage, no parameters are trainable yet.

In [10]:
%pip install mistral-common
%pip install bitsandbytes

In [11]:
# ============================================================
# Load Voxtral Model & Processor
# ============================================================

from transformers import AutoProcessor, VoxtralForConditionalGeneration
import torch

LOCAL_MODEL_DIR = "/content/drive/MyDrive/adsp/models/voxtral-mini-3b"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

processor = AutoProcessor.from_pretrained(
    LOCAL_MODEL_DIR,
    trust_remote_code=True
)

model = VoxtralForConditionalGeneration.from_pretrained(
    LOCAL_MODEL_DIR,
    trust_remote_code=True,
    load_in_8bit=True,
    device_map="auto"
)

model.train()

Device: cpu


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

VoxtralForConditionalGeneration(
  (audio_tower): VoxtralEncoder(
    (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 1280)
    (layers): ModuleList(
      (0-31): 32 x VoxtralEncoderLayer(
        (self_attn): VoxtralAttention(
          (k_proj): Linear8bitLt(in_features=1280, out_features=1280, bias=False)
          (v_proj): Linear8bitLt(in_features=1280, out_features=1280, bias=True)
          (q_proj): Linear8bitLt(in_features=1280, out_features=1280, bias=True)
          (out_proj): Linear8bitLt(in_features=1280, out_features=1280, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear8bitLt(in_features=1280, out_features=5120, bias=True)
        (fc2): Linear8bitLt(in_features=5120, out_features=1280, bias=True)
        (fina

## Data Collator for Instruction Tuning

The data collator is responsible for transforming dataset examples into model inputs.

Steps performed for each batch:
1. Decode the stored JSON conversation
2. Apply Voxtral’s chat template
3. Tokenize the target emotion label
4. Concatenate prompt and label tokens
5. Mask prompt tokens using `-100` so that loss is computed only on the label

This setup implements **causal language modeling loss** for instruction tuning.

In [12]:
# ============================================================
# Data Collator (JSON-safe, instruction tuning)
# Builds input_ids, attention_mask, labels with prompt-masked loss.
# ============================================================

import torch
import json

class VoxtralInstructionCollator:
    def __init__(self, processor, device):
        self.processor = processor
        self.device = device

    def __call__(self, batch):
        input_ids_list = []
        attn_list = []
        labels_list = []

        for ex in batch:
            conversation = ex["conversation"]
            if isinstance(conversation, str):
                conversation = json.loads(conversation)

            prompt = self.processor.apply_chat_template(
                conversation,
                tokenize=True,
                return_tensors="pt"
            )

            # Label tokens (no special tokens)
            label_ids = self.processor.tokenizer(
                ex["label"],
                add_special_tokens=False,
                return_tensors="pt"
            ).input_ids

            input_ids = torch.cat([prompt["input_ids"], label_ids], dim=1)

            # attention mask: 1 for real tokens
            prompt_attn = prompt.get("attention_mask", torch.ones_like(prompt["input_ids"]))
            label_attn = torch.ones_like(label_ids)
            attention_mask = torch.cat([prompt_attn, label_attn], dim=1)

            # mask prompt tokens in loss
            labels = torch.cat(
                [torch.full_like(prompt["input_ids"], -100), label_ids],
                dim=1
            )

            input_ids_list.append(input_ids.squeeze(0))
            attn_list.append(attention_mask.squeeze(0))
            labels_list.append(labels.squeeze(0))

        input_ids_padded = torch.nn.utils.rnn.pad_sequence(
            input_ids_list,
            batch_first=True,
            padding_value=self.processor.tokenizer.pad_token_id
        )
        attn_padded = torch.nn.utils.rnn.pad_sequence(
            attn_list,
            batch_first=True,
            padding_value=0
        )
        labels_padded = torch.nn.utils.rnn.pad_sequence(
            labels_list,
            batch_first=True,
            padding_value=-100
        )

        batch_inputs = {
            "input_ids": input_ids_padded,
            "attention_mask": attn_padded,
            "labels": labels_padded,
        }
        return {k: v.to(self.device) for k, v in batch_inputs.items()}

data_collator = VoxtralInstructionCollator(processor, device)

In [13]:
print(type(train_ds[0]["conversation"]))
print(train_ds[0]["conversation"][:80])  # should print a JSON string prefix

<class 'str'>
[{"role": "system", "content": "You are a careful and concise emotion classifica


In [14]:
# ============================================================
# Dummy Test 2: Data collator sanity check
# ============================================================

batch = [train_ds[i] for i in range(2)]
out = data_collator(batch)

for k, v in out.items():
    print(k, v.shape, v.dtype, v.device)

input_ids torch.Size([2, 424]) torch.int64 cpu
attention_mask torch.Size([2, 424]) torch.int64 cpu
labels torch.Size([2, 424]) torch.int64 cpu


In [ ]:
# ============================================================
# Dummy Test 3: Single forward + backward pass
# ============================================================

model.train()

batch = [train_ds[0]]
out = data_collator(batch)

loss = model(**out).loss
print("Loss:", loss.item())

loss.backward()
print("Backward pass OK")

Loss: 4.064279079437256


## Parameter-Efficient Fine-Tuning with LoRA

To keep training lightweight, we use **LoRA (Low-Rank Adaptation)**:

- The base Voxtral model remains frozen
- Only a small number of adapter parameters are trained
- This drastically reduces memory and compute requirements

We apply LoRA to the query and value projection layers of the language model.

In [ ]:
# ============================================================
# Configure PEFT (LoRA)
# ============================================================

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training Configuration and Logging

We configure the Hugging Face `Trainer` with:

- Small per-device batch size
- Gradient accumulation to simulate a larger batch
- Step-based logging
- TensorBoard support
- Epoch-based checkpointing

The training objective is to minimize the language modeling loss on the generated emotion label.

In [1]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./voxtral-esd",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,              # ✅ real training
    fp16=torch.cuda.is_available(),

    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=25,

    evaluation_strategy="epoch",     # ✅ same idea as before, less frequent
    save_strategy="epoch",           # ✅ same idea as before, less frequent
    save_total_limit=2,

    report_to=["tensorboard"],
    remove_unused_columns=False,
    dataloader_pin_memory=False,     # ✅ keep the crash fix
)

NameError: name 'TrainingArguments' is not defined

In [ ]:
# ============================================================
# Dummy Test 4: Tiny Trainer run (5 steps)
# ============================================================

tiny_train = train_ds.select(range(8))
tiny_test  = test_ds.select(range(8))

dummy_args = TrainingArguments(
    output_dir="./dummy-run",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_steps=5,
    logging_steps=1,
    evaluation_strategy="steps",
    eval_steps=5,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

dummy_trainer = Trainer(
    model=model,
    args=dummy_args,
    train_dataset=tiny_train,
    eval_dataset=tiny_test,
    data_collator=data_collator,
)

dummy_trainer.train()

In [ ]:
# ============================================================
# Dummy Test 5: Save & reload adapters
# ============================================================

model.save_pretrained("./dummy-adapters")

from peft import PeftModel
reloaded = PeftModel.from_pretrained(model.base_model, "./dummy-adapters")
reloaded.eval()

print("Adapter save & reload OK")

## Training Time Monitoring

We add a custom Trainer callback to monitor:
- Elapsed training time
- Estimated remaining time (ETA)
- Loss progression

This provides feedback similar to progress indicators used in software installation or game downloads.

In [27]:
# ============================================================
# Trainer Time Logging Callback
# ============================================================

import time
from transformers import TrainerCallback


class TimeLoggingCallback(TrainerCallback):
    def __init__(self):
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("⏱️ Training started...")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.start_time is None or logs is None:
            return

        # Only handle training loss logs
        if "loss" not in logs:
            return

        elapsed = time.time() - self.start_time
        elapsed_str = time.strftime("%H:%M:%S", time.gmtime(elapsed))

        steps_done = state.global_step
        total_steps = state.max_steps

        time_per_step = elapsed / max(steps_done, 1)
        remaining = (total_steps - steps_done) * time_per_step
        remaining_str = time.strftime("%H:%M:%S", time.gmtime(remaining))

        loss = logs["loss"]

        print(
            f"[Step {steps_done}/{total_steps}] "
            f"loss={loss:.4f} | elapsed={elapsed_str} | ETA={remaining_str}"
        )

## Running Fine-Tuning

We now launch the fine-tuning process.

Before training starts, the Trainer prints:
- Number of training examples
- Number of epochs
- Total optimization steps
- Number of trainable parameters

These values are useful to verify that:
- LoRA is correctly applied
- Training is not accidentally skipped

In [22]:
# ============================================================
# Trainer & Training
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    callbacks=[TimeLoggingCallback()],
)

trainer.train()

max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
***** Running training *****
  Num examples = 9,800
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 100
  Number of trainable parameters = 4,014,080


⏱️ Training started...


Epoch,Training Loss,Validation Loss


[Step 10/100] loss=0.3731 | elapsed=00:01:07 | ETA=00:10:06
[Step 20/100] loss=0.3588 | elapsed=00:02:15 | ETA=00:09:02
[Step 30/100] loss=0.3735 | elapsed=00:03:25 | ETA=00:07:58
[Step 40/100] loss=0.3504 | elapsed=00:04:34 | ETA=00:06:51
[Step 50/100] loss=0.3609 | elapsed=00:05:54 | ETA=00:05:54
[Step 60/100] loss=0.3644 | elapsed=00:07:31 | ETA=00:05:00
[Step 70/100] loss=0.3696 | elapsed=00:09:06 | ETA=00:03:54
[Step 80/100] loss=0.3622 | elapsed=00:10:41 | ETA=00:02:40
[Step 90/100] loss=0.3507 | elapsed=00:12:17 | ETA=00:01:21



***** Running Evaluation *****
  Num examples = 4200
  Batch size = 8


[Step 100/100] loss=0.3333 | elapsed=00:13:54 | ETA=00:00:00


TypeError: unsupported format string passed to NoneType.__format__

## Saving the LoRA Adapters

After fine-tuning, we save only the trained LoRA adapters. This allows us to share and load the lightweight adapters separately from the large base model.

In [23]:
model.save_pretrained("./voxtral-esd-optionA-adapters")
print("Adapters saved.")

loading configuration file /content/drive/MyDrive/adsp/models/voxtral-mini-3b/config.json
Model config VoxtralConfig {
  "architectures": [
    "VoxtralForConditionalGeneration"
  ],
  "audio_config": {
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "attention_dropout": 0.0,
    "dropout": 0.0,
    "head_dim": 64,
    "hidden_size": 1280,
    "initializer_range": 0.02,
    "intermediate_size": 5120,
    "layerdrop": 0.0,
    "max_source_positions": 1500,
    "model_type": "voxtral_encoder",
    "num_attention_heads": 20,
    "num_hidden_layers": 32,
    "num_key_value_heads": 20,
    "num_mel_bins": 128,
    "scale_embedding": false,
    "vocab_size": 51866
  },
  "audio_token_id": 24,
  "dtype": "bfloat16",
  "hidden_size": 3072,
  "model_type": "voxtral",
  "projector_hidden_act": "gelu",
  "text_config": {
    "attention_bias": false,
    "attention_dropout": 0.0,
    "head_dim": 128,
    "hidden_act": "silu",
    "hidden_size": 3072,
    "initializer_range": 

Adapters saved.


## Loading the Fine-Tuned Model for Inference

To prepare for inference, we load the base **Voxtral-mini-3B** model and then merge the newly trained LoRA adapters onto it. The model is set to evaluation mode (`model.eval()`).

In [28]:
from transformers import VoxtralForConditionalGeneration, AutoProcessor
from peft import PeftModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_MODEL_DIR = "/content/drive/MyDrive/adsp/models/voxtral-mini-3b"
ADAPTER_DIR = "./voxtral-esd-optionA-adapters"

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_DIR,
    trust_remote_code=True
)

base_model = VoxtralForConditionalGeneration.from_pretrained(
    BASE_MODEL_DIR,
    trust_remote_code=True,
    load_in_8bit=True,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

loading configuration file /content/drive/MyDrive/adsp/models/voxtral-mini-3b/preprocessor_config.json
loading configuration file /content/drive/MyDrive/adsp/models/voxtral-mini-3b/preprocessor_config.json
Feature extractor WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 128,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "processor_class": "VoxtralProcessor",
  "return_attention_mask": false,
  "sampling_rate": 16000
}

loading configuration file None
Processor VoxtralProcessor:
- feature_extractor: WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 128,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "processor_class": "VoxtralProce

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loading configuration file /content/drive/MyDrive/adsp/models/voxtral-mini-3b/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 2,
  "pad_token_id": 11
}

Could not locate the custom_generate/generate.py inside /content/drive/MyDrive/adsp/models/voxtral-mini-3b.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): VoxtralForConditionalGeneration(
      (audio_tower): VoxtralEncoder(
        (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
        (embed_positions): Embedding(1500, 1280)
        (layers): ModuleList(
          (0-31): 32 x VoxtralEncoderLayer(
            (self_attn): VoxtralAttention(
              (k_proj): Linear8bitLt(in_features=1280, out_features=1280, bias=False)
              (v_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1280, out_features=1280, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1280, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (defau

## Qualitative Inference on Test Samples

We perform a qualitative evaluation by running inference on a small, random subset of the test dataset. For each sample, we print the ground truth label and the model's prediction to get a sense of its performance.

In [35]:
import random

for i in random.sample(range(len(test_ds)), 30):
    sample = test_ds[i]

    conversation = json.loads(sample["conversation"])

    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=3)

    pred = processor.batch_decode(
        out[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )[0]

    print(f"Sample {i}")
    print("GT :", sample["label"])
    print("PR :", pred)
    print("-" * 30)

Sample 938
GT : Happy
PR : Neutral
------------------------------
Sample 2185
GT : Happy
PR : Neutral
------------------------------
Sample 2786
GT : Sad
PR : Neutral
------------------------------
Sample 913
GT : Happy
PR : Neutral
------------------------------
Sample 2404
GT : Happy
PR : Neutral
------------------------------
Sample 3561
GT : Happy
PR : Neutral
------------------------------
Sample 1295
GT : Sad
PR : Neutral
------------------------------
Sample 3716
GT : Happy
PR : Angry
------------------------------
Sample 26
GT : Neutral
PR : Neutral
------------------------------
Sample 2157
GT : Happy
PR : Neutral
------------------------------
Sample 4100
GT : Sad
PR : Neutral
------------------------------
Sample 1463
GT : Neutral
PR : Neutral
------------------------------
Sample 4158
GT : Sad
PR : Neutral
------------------------------
Sample 871
GT : Happy
PR : Neutral
------------------------------
Sample 2444
GT : Happy
PR : Neutral
------------------------------
Sample